# Paper Figure 3 — PCA vs MS-AA simulation benchmark 

This notebook is the manuscript-figure companion to
`revision_16_pca_msaa_clean_simulation_benchmark.ipynb`.

It does **not** rerun the full 20-replicate benchmark. Instead it:
1. loads the paper-mode checkpoint produced by the canonical simulation notebook;
2. recomputes one objectively preselected representative metastable replicate
   (seed `1941156788`) for the example-pattern panel;
3. rebuilds the main manuscript simulation figure.

The main distinction is between:
- direct recovery of individual recurring configurations; and
- preservation of the full low-dimensional latent span.

The figure should not be interpreted as claiming universal superiority of AA
over PCA.

### Correction in this version

The results checkpoint uses the regime identifiers `PCA_control`, `AA_control`,
and `metastable`. An earlier paper-figure notebook incorrectly looked for
`pca_favorable` and `aa_favorable`, which caused the two calibration-control
groups to disappear from Panel A. This version uses the canonical identifiers
from the simulation analysis and reproduces the intended paired-difference
Panel B.


In [ ]:
import numpy as np

# ============================================================
# Configuration
# ============================================================

RUN_MODE = "paper"   # "quick" or "paper"
RNG_SEED = 42

HELPERS_DIR = "helpers"
HYPERTOOLS_NORMALIZE_PATH = "hypertools/normalize.py"

# New checkpoint namespace: main analysis is now PCA vs MS-AA only.
OUTPUT_DIR = "simulation_pca_msaa_extended_outputs"

ANALYSIS_TYPE = "spatial"

MSAA_OPTS_BASE = dict(
    maxiter=200,
    conv_crit=1e-6,
    fix_var_iter=5,
    heteroscedastic=False,
    use_gpu=False,
    rngSEED=RNG_SEED,
)

# Shared spatial architecture
N_NETWORKS = 6
SUBNETWORKS_PER_NETWORK = 2
NODES_PER_SUBNETWORK = 10
N_TRUE_MOTIFS = 6

BASE_NOISE_SD = 0.25
BASE_MULTIPLICATIVE_SD = 0.08
BASE_ADDITIVE_SD = 0.04

# PCA-positive-control source standard deviations.
PCA_SOURCE_SCALES = np.array(
    [2.0, 1.7, 1.4, 1.1, 0.8, 0.5]
)

# AA-positive-control geometry
DIRICHLET_ALPHA = 0.10

# Brain-inspired metastable dynamics
MEAN_DWELL = 12
TRANSITION_WIDTH = 4
SECONDARY_WEIGHT = 0.25
AR1_RHO = 0.85

# Make metastable dynamics deliberately non-simplex-exact.
GLOBAL_GAIN_SD = 0.15
SIGNED_DEVIATION_SD = 0.08
DEVIATION_RHO = 0.80

# Motif-overlap manipulation.
# Higher distinctiveness -> lower overlap.
OVERLAP_LEVELS = {
    "low_overlap": 1.15,
    "baseline_overlap": 1.00,
    "high_overlap": 0.55,
}

# Joint dynamics manipulation.
DYNAMICS_PRESETS = {
    "state_like": {
        "mean_dwell": 16,
        "transition_width": 2,
        "secondary_weight": 0.10,
        "ar1_rho": 0.75,
    },
    "baseline": {
        "mean_dwell": 12,
        "transition_width": 4,
        "secondary_weight": 0.25,
        "ar1_rho": 0.85,
    },
    "more_mixed": {
        "mean_dwell": 8,
        "transition_width": 8,
        "secondary_weight": 0.45,
        "ar1_rho": 0.90,
    },
}

# Secondary robustness
NOISE_LEVELS = [0.10, 0.25, 0.40]
HETEROGENEITY_LEVELS = [0.0, 1.0, 1.5]
K_LEVELS = [4, 6, 8]

# Optional supplementary ICA calibration.
# This is OFF by default and never enters the main checkpoint/results.
RUN_OPTIONAL_ICA_CALIBRATION = False

if RUN_MODE == "quick":
    N_SUBJECTS = 10
    T_PER_SUBJECT = 120
    N_REPS_CORE = 2
    N_REPS_METASTABLE = 3
    N_REPS_ROBUST = 1
    N_REPS_SENSITIVITY = 1
    OPTIONAL_ICA_REPS = 2
elif RUN_MODE == "paper":
    N_SUBJECTS = 16
    T_PER_SUBJECT = 180
    N_REPS_CORE = 20
    N_REPS_METASTABLE = 20
    N_REPS_ROBUST = 10
    N_REPS_SENSITIVITY = 10
    OPTIONAL_ICA_REPS = 5
else:
    raise ValueError(
        "RUN_MODE must be 'quick' or 'paper'"
    )

print("RUN_MODE:", RUN_MODE)
print("Subjects:", N_SUBJECTS)
print("Time points per subject:", T_PER_SUBJECT)
print("Core reps:", N_REPS_CORE)
print("Metastable reps:", N_REPS_METASTABLE)
print("Robustness reps:", N_REPS_ROBUST)
print("Sensitivity reps:", N_REPS_SENSITIVITY)
print("Optional ICA calibration:", RUN_OPTIONAL_ICA_CALIBRATION)


In [ ]:
# ============================================================
# Imports
# ============================================================

%matplotlib inline

import os
import sys
import time
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import PCA

sys.path.append(HELPERS_DIR)
from MultiSubject_AA import Subject, multi_subject_aa

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RESULTS_CSV = OUTPUT_DIR / (
    f"pca_msaa_extended_results_{RUN_MODE}.csv"
)

print("Checkpoint:", RESULTS_CSV)


In [ ]:
def coerce_TxV(x):
    x = np.asarray(x, dtype=float)

    if x.ndim != 2:
        raise ValueError(f"Expected 2D T x V matrix, got {x.shape}")

    x = x - x.mean(axis=0, keepdims=True)
    return x.astype(np.float32, copy=False)


def orient_subject_matrix(X_raw, analysis_type):
    Xtv = coerce_TxV(X_raw)
    return Xtv if analysis_type == "spatial" else Xtv.T


def build_subject_objects(subject_list, analysis_type):
    return [
        Subject(
            X=orient_subject_matrix(x, analysis_type),
            sX=orient_subject_matrix(x, analysis_type),
        )
        for x in subject_list
    ]


def fit_msaa(subject_list, K, analysis_type="spatial", seed=RNG_SEED, verbose=False):
    opts = dict(MSAA_OPTS_BASE)
    opts["rngSEED"] = int(seed)

    subj_objs = build_subject_objects(subject_list, analysis_type)

    t0 = time.time()

    results_subj, C, cost_fun, varexpl, secs = multi_subject_aa(
        subj_objs,
        noc=int(K),
        opts=opts,
    )

    elapsed = time.time() - t0

    if verbose:
        print(
            f"[MS-AA] K={K} | VarExpl={varexpl*100:.2f}% | "
            f"{elapsed:.2f}s"
        )

    return {
        "results_subj": results_subj,
        "C": C,
        "cost_fun": cost_fun,
        "varexpl": varexpl,
        "secs": secs,
        "elapsed": elapsed,
        "K": int(K),
    }


def reconstruct_from_archetypes(sub, analysis_type="spatial"):
    sXC = np.asarray(sub["sXC"], dtype=float)
    S = np.asarray(sub["S"], dtype=float)

    if analysis_type == "spatial":
        return sXC @ S

    return (sXC @ S).T


In [ ]:
def make_overlapping_motifs():
    n_subnetworks = (
        N_NETWORKS
        * SUBNETWORKS_PER_NETWORK
    )

    V = (
        n_subnetworks
        * NODES_PER_SUBNETWORK
    )

    motifs = np.full(
        (N_TRUE_MOTIFS, V),
        0.05,
        dtype=float,
    )

    motif_subnetworks = [
        [0, 3, 8],
        [1, 3, 9],
        [2, 5, 10],
        [2, 4, 11],
        [0, 6, 7],
        [1, 6, 8],
    ]

    for k, active in enumerate(
        motif_subnetworks
    ):
        for subnet in active:
            lo = (
                subnet
                * NODES_PER_SUBNETWORK
            )
            hi = (
                lo
                + NODES_PER_SUBNETWORK
            )

            motifs[
                k,
                lo:hi,
            ] = 1.0

    return motifs


def make_overlap_variant(
    base_motifs,
    distinctiveness_scale,
):
    # Scale each motif's deviation from the common motif.
    # This preserves network topology while changing
    # nonorthogonality/overlap.
    base_motifs = np.asarray(
        base_motifs,
        dtype=float,
    )

    common = base_motifs.mean(
        axis=0,
        keepdims=True,
    )

    return (
        common
        + distinctiveness_scale
        * (
            base_motifs
            - common
        )
    )


def mean_pairwise_cosine(patterns):
    patterns = np.asarray(
        patterns,
        dtype=float,
    )

    unit = patterns / (
        np.linalg.norm(
            patterns,
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    C = unit @ unit.T

    tri = np.triu_indices(
        C.shape[0],
        k=1,
    )

    return float(
        np.mean(C[tri])
    )


TRUE_MOTIFS_RAW = (
    make_overlapping_motifs()
)

V = TRUE_MOTIFS_RAW.shape[1]

print(
    "Motif matrix:",
    TRUE_MOTIFS_RAW.shape,
)

for name, scale in (
    OVERLAP_LEVELS.items()
):
    variant = make_overlap_variant(
        TRUE_MOTIFS_RAW,
        scale,
    )

    print(
        name,
        "| distinctiveness =",
        scale,
        "| mean pairwise cosine =",
        round(
            mean_pairwise_cosine(
                variant
            ),
            3,
        ),
    )


In [ ]:
def plot_pattern_matrix(patterns, title, ylabel="Pattern"):
    fig, ax = plt.subplots(figsize=(12, 4))
    im = ax.imshow(patterns, aspect="auto")

    for x in range(
        NODES_PER_SUBNETWORK,
        patterns.shape[1],
        NODES_PER_SUBNETWORK,
    ):
        ax.axvline(x - 0.5, linewidth=0.4, alpha=0.35)

    network_width = (
        SUBNETWORKS_PER_NETWORK
        * NODES_PER_SUBNETWORK
    )

    for x in range(
        network_width,
        patterns.shape[1],
        network_width,
    ):
        ax.axvline(x - 0.5, linewidth=1.2)

    ax.set_xlabel("Nodes grouped by network → subnetwork")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.8)

    plt.show()


plot_pattern_matrix(
    TRUE_MOTIFS_RAW,
    "Ground-truth overlapping distributed network motifs",
    ylabel="True motif",
)


In [ ]:
def normalize_across_subjects(
    subject_list,
    eps=1e-8,
):
    # Match the empirical fallback:
    # coerce/center EACH subject first,
    # then scale features across the stacked subjects.
    raw = [
        np.asarray(
            x,
            dtype=float,
        )
        for x in subject_list
    ]

    subject_means = [
        x.mean(
            axis=0,
            keepdims=True,
        )
        for x in raw
    ]

    centered = [
        x - mu_s
        for x, mu_s in zip(
            raw,
            subject_means,
        )
    ]

    stacked = np.vstack(
        centered
    )

    across_mu = stacked.mean(
        axis=0,
        keepdims=True,
    )

    sd = (
        stacked.std(
            axis=0,
            keepdims=True,
        )
        + eps
    )

    normalized = [
        (
            (x - across_mu)
            / sd
        ).astype(
            np.float32,
            copy=False,
        )
        for x in centered
    ]

    mean_subject_mean = np.mean(
        np.stack(
            subject_means,
            axis=0,
        ),
        axis=0,
    )

    return (
        normalized,
        mean_subject_mean,
        sd,
    )


def transform_true_patterns(
    true_patterns_raw,
    mean_subject_mean,
    sd,
    truth_kind,
):
    true_patterns_raw = np.asarray(
        true_patterns_raw,
        dtype=float,
    )

    if truth_kind == "direction":
        # Centering observations changes origins,
        # not generative directions.
        return (
            true_patterns_raw
            / sd
        )

    if truth_kind == "point":
        # State/archetype templates are points.
        # Averaging subject-wise affine transforms
        # gives this group-level transform.
        return (
            (
                true_patterns_raw
                - mean_subject_mean
            )
            / sd
        )

    raise ValueError(
        "truth_kind must be "
        "'direction' or 'point'"
    )


def pattern_corr_matrix(
    A,
    B,
    absolute=False,
):
    A = np.asarray(
        A,
        dtype=float,
    )

    B = np.asarray(
        B,
        dtype=float,
    )

    A0 = (
        A
        - A.mean(
            axis=1,
            keepdims=True,
        )
    )

    B0 = (
        B
        - B.mean(
            axis=1,
            keepdims=True,
        )
    )

    A0 /= (
        np.linalg.norm(
            A0,
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    B0 /= (
        np.linalg.norm(
            B0,
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    C = A0 @ B0.T

    return (
        np.abs(C)
        if absolute
        else C
    )


def optimal_match(
    true_patterns,
    recovered_patterns,
    absolute=False,
):
    C = pattern_corr_matrix(
        true_patterns,
        recovered_patterns,
        absolute=absolute,
    )

    (
        true_idx,
        recovered_idx,
    ) = linear_sum_assignment(
        -C
    )

    return {
        "similarity": C,
        "true_idx": true_idx,
        "recovered_idx": recovered_idx,
        "matched": C[
            true_idx,
            recovered_idx,
        ],
    }


def coverage_adjusted_recovery(
    match,
    n_true,
):
    return float(
        np.sum(
            match["matched"]
        )
        / n_true
    )



def temporal_recovery_from_spatial_match(
    true_activations,
    recovered_scores,
    spatial_match,
    absolute=False,
):
    # Temporal recovery is anchored to spatial matching.
    # Because simulated observations are centered within subject,
    # center latent/recovered trajectories within subject before
    # concatenating across subjects.
    if true_activations is None:
        return {
            "mean": np.nan,
            "coverage": np.nan,
            "per_pattern": (
                np.array([])
            ),
        }

    true_list = [
        np.asarray(
            a,
            dtype=float,
        )
        for a in true_activations
    ]

    true_centered = [
        a - a.mean(
            axis=0,
            keepdims=True,
        )
        for a in true_list
    ]

    true_stack = np.vstack(
        true_centered
    )

    if isinstance(
        recovered_scores,
        (list, tuple),
    ):
        recovered_list = [
            np.asarray(
                a,
                dtype=float,
            )
            for a in recovered_scores
        ]
    else:
        recovered_array = np.asarray(
            recovered_scores,
            dtype=float,
        )

        lengths = [
            a.shape[0]
            for a in true_list
        ]

        cuts = np.cumsum(
            lengths
        )[:-1]

        recovered_list = (
            np.split(
                recovered_array,
                cuts,
                axis=0,
            )
        )

    recovered_centered = [
        a - a.mean(
            axis=0,
            keepdims=True,
        )
        for a in recovered_list
    ]

    recovered_stack = np.vstack(
        recovered_centered
    )

    vals = []

    for (
        true_idx,
        recovered_idx,
    ) in zip(
        spatial_match["true_idx"],
        spatial_match[
            "recovered_idx"
        ],
    ):
        a = true_stack[
            :,
            true_idx,
        ]

        b = recovered_stack[
            :,
            recovered_idx,
        ]

        if (
            np.std(a) < 1e-12
            or np.std(b) < 1e-12
        ):
            r = np.nan
        else:
            r = np.corrcoef(
                a,
                b,
            )[0, 1]

        if (
            absolute
            and np.isfinite(r)
        ):
            r = abs(r)

        vals.append(r)

    vals = np.asarray(
        vals,
        dtype=float,
    )

    return {
        "mean": float(
            np.nanmean(vals)
        ),
        "coverage": float(
            np.nansum(vals)
            / true_stack.shape[1]
        ),
        "per_pattern": vals,
    }

def collapse_to_subnetworks(
    patterns,
):
    patterns = np.asarray(
        patterns,
        dtype=float,
    )

    n_subnetworks = (
        N_NETWORKS
        * SUBNETWORKS_PER_NETWORK
    )

    out = np.zeros(
        (
            patterns.shape[0],
            n_subnetworks,
        )
    )

    for subnet in range(
        n_subnetworks
    ):
        lo = (
            subnet
            * NODES_PER_SUBNETWORK
        )

        hi = (
            lo
            + NODES_PER_SUBNETWORK
        )

        out[
            :,
            subnet,
        ] = (
            patterns[
                :,
                lo:hi,
            ].mean(
                axis=1
            )
        )

    return out


def linear_span_recovery(
    true_patterns,
    recovered_patterns,
):
    # Project true patterns onto the row span
    # of the recovered patterns.
    B = np.asarray(
        recovered_patterns,
        dtype=float,
    ).T

    vals = []

    for true in np.asarray(
        true_patterns,
        dtype=float,
    ):
        coef, *_ = np.linalg.lstsq(
            B,
            true,
            rcond=None,
        )

        recon = (
            B @ coef
        )

        vals.append(
            np.corrcoef(
                true,
                recon,
            )[0, 1]
        )

    return float(
        np.nanmean(vals)
    )


In [ ]:
def evaluate_methods(
    subject_data,
    true_patterns_raw,
    K_fit,
    seed,
    regime,
    rep,
    true_activations=None,
    truth_kind="point",
    extra=None,
    verbose=False,
):
    """
    Matched PCA vs MS-AA evaluation.

    Primary outcomes:
      - direct spatial pattern recovery
      - temporal activation recovery anchored to the spatial match
      - subnetwork-composition recovery
      - latent-span recovery
      - reconstruction MSE

    PCA spatial/temporal correlations are sign-invariant because PCA axes have
    arbitrary sign. MS-AA uses signed correlations.
    """
    (
        subjects,
        mean_subject_mean,
        sd,
    ) = normalize_across_subjects(
        subject_data
    )

    true_patterns = transform_true_patterns(
        true_patterns_raw,
        mean_subject_mean,
        sd,
        truth_kind=truth_kind,
    )

    X_stack = np.vstack(
        subjects
    )

    # ========================================================
    # MS-AA
    # ========================================================

    msaa = fit_msaa(
        subjects,
        K=K_fit,
        analysis_type="spatial",
        seed=seed,
        verbose=verbose,
    )

    S_group = np.mean(
        np.stack(
            [
                np.asarray(
                    sub["S"],
                    dtype=float,
                )
                for sub in msaa[
                    "results_subj"
                ]
            ],
            axis=0,
        ),
        axis=0,
    )

    aa_scores = [
        np.asarray(
            sub["sXC"],
            dtype=float,
        )
        for sub in msaa[
            "results_subj"
        ]
    ]

    Xhat_aa = np.vstack([
        reconstruct_from_archetypes(
            sub,
            "spatial",
        )
        for sub in msaa[
            "results_subj"
        ]
    ])

    aa_mse = float(
        np.mean(
            (X_stack - Xhat_aa) ** 2
        )
    )

    # ========================================================
    # PCA
    # ========================================================

    pca = PCA(
        n_components=int(K_fit),
        random_state=int(seed),
    )

    pca_scores = pca.fit_transform(
        X_stack
    )

    Xhat_pca = pca.inverse_transform(
        pca_scores
    )

    pca_patterns = np.asarray(
        pca.components_,
        dtype=float,
    )

    pca_mse = float(
        np.mean(
            (X_stack - Xhat_pca) ** 2
        )
    )

    # ========================================================
    # Spatial recovery
    # ========================================================

    aa_match = optimal_match(
        true_patterns,
        S_group,
        absolute=False,
    )

    pca_match = optimal_match(
        true_patterns,
        pca_patterns,
        absolute=True,
    )

    # ========================================================
    # Temporal recovery anchored to the spatial match
    # ========================================================

    aa_temporal = temporal_recovery_from_spatial_match(
        true_activations,
        aa_scores,
        aa_match,
        absolute=False,
    )

    pca_temporal = temporal_recovery_from_spatial_match(
        true_activations,
        pca_scores,
        pca_match,
        absolute=True,
    )

    # ========================================================
    # Subnetwork recovery
    # ========================================================

    true_sub = collapse_to_subnetworks(
        true_patterns
    )

    aa_sub = collapse_to_subnetworks(
        S_group
    )

    pca_sub = collapse_to_subnetworks(
        pca_patterns
    )

    aa_sub_match = optimal_match(
        true_sub,
        aa_sub,
        absolute=False,
    )

    pca_sub_match = optimal_match(
        true_sub,
        pca_sub,
        absolute=True,
    )

    # ========================================================
    # Span recovery
    # ========================================================

    aa_span = linear_span_recovery(
        true_patterns,
        S_group,
    )

    pca_span = linear_span_recovery(
        true_patterns,
        pca_patterns,
    )

    n_true = true_patterns.shape[0]

    row = {
        "regime": regime,
        "condition": "base",
        "rep": int(rep),
        "seed": int(seed),
        "K_fit": int(K_fit),
        "truth_kind": truth_kind,

        "aa_motif_recovery": float(
            np.mean(
                aa_match["matched"]
            )
        ),
        "pca_motif_recovery": float(
            np.mean(
                pca_match["matched"]
            )
        ),

        "aa_motif_recovery_coverage": (
            coverage_adjusted_recovery(
                aa_match,
                n_true,
            )
        ),
        "pca_motif_recovery_coverage": (
            coverage_adjusted_recovery(
                pca_match,
                n_true,
            )
        ),

        "aa_temporal_recovery": (
            aa_temporal["mean"]
        ),
        "pca_temporal_recovery": (
            pca_temporal["mean"]
        ),

        "aa_temporal_recovery_coverage": (
            aa_temporal["coverage"]
        ),
        "pca_temporal_recovery_coverage": (
            pca_temporal["coverage"]
        ),

        "aa_subnetwork_recovery": float(
            np.mean(
                aa_sub_match["matched"]
            )
        ),
        "pca_subnetwork_recovery": float(
            np.mean(
                pca_sub_match["matched"]
            )
        ),

        "aa_span_recovery": aa_span,
        "pca_span_recovery": pca_span,

        "aa_mse": aa_mse,
        "pca_mse": pca_mse,

        "aa_varexpl": float(
            msaa["varexpl"]
        ),
        "pca_explained_variance": float(
            np.sum(
                pca.explained_variance_ratio_
            )
        ),

        "msaa_elapsed_sec": float(
            msaa["elapsed"]
        ),
    }

    if extra:
        row.update(extra)

    return row


In [ ]:
def simulate_pca_control(seed):
    rng = np.random.default_rng(
        seed
    )

    Q, _ = np.linalg.qr(
        rng.normal(
            size=(
                V,
                N_TRUE_MOTIFS,
            )
        )
    )

    true_factors = Q.T

    subjects = []
    activations = []

    for s in range(
        N_SUBJECTS
    ):
        scores = rng.normal(
            size=(
                T_PER_SUBJECT,
                N_TRUE_MOTIFS,
            )
        )

        scores = (
            scores
            * PCA_SOURCE_SCALES[
                None,
                :,
            ]
        )

        X = (
            scores
            @ true_factors
        )

        X += rng.normal(
            0,
            BASE_NOISE_SD,
            size=X.shape,
        )

        subjects.append(X)
        activations.append(
            scores
        )

    return {
        "subjects": subjects,
        "true_patterns": (
            true_factors
        ),
        "activations": (
            activations
        ),
        "truth_kind": "direction",
    }


In [ ]:
def simulate_aa_control(
    seed,
):
    rng = np.random.default_rng(
        seed
    )

    subjects = []
    activations = []

    for s in range(
        N_SUBJECTS
    ):
        multiplicative = (
            1.0
            + rng.normal(
                0,
                BASE_MULTIPLICATIVE_SD,
                size=(
                    TRUE_MOTIFS_RAW.shape
                ),
            )
        )

        additive = rng.normal(
            0,
            BASE_ADDITIVE_SD,
            size=(
                TRUE_MOTIFS_RAW.shape
            ),
        )

        subj_motifs = (
            TRUE_MOTIFS_RAW
            * multiplicative
            + additive
        )

        W = rng.dirichlet(
            np.full(
                N_TRUE_MOTIFS,
                DIRICHLET_ALPHA,
            ),
            size=T_PER_SUBJECT,
        )

        X = (
            W @ subj_motifs
        )

        X += rng.normal(
            0,
            BASE_NOISE_SD,
            size=X.shape,
        )

        subjects.append(X)
        activations.append(W)

    return {
        "subjects": subjects,
        "true_patterns": (
            TRUE_MOTIFS_RAW
        ),
        "activations": (
            activations
        ),
        "truth_kind": "point",
    }


In [ ]:
def sample_state_sequence(
    T,
    K,
    rng,
    mean_dwell=MEAN_DWELL,
):
    states = []
    current = int(
        rng.integers(K)
    )

    while len(states) < T:
        dwell = max(
            2,
            int(
                rng.geometric(
                    1.0
                    / mean_dwell
                )
            ),
        )

        states.extend(
            [current] * dwell
        )

        candidates = [
            k
            for k in range(K)
            if k != current
        ]

        current = int(
            rng.choice(
                candidates
            )
        )

    return np.asarray(
        states[:T],
        dtype=int,
    )


def smooth_state_activations(
    states,
    K,
    transition_width=(
        TRANSITION_WIDTH
    ),
):
    T = len(states)

    A = np.zeros(
        (T, K),
        dtype=float,
    )

    A[
        np.arange(T),
        states,
    ] = 1.0

    if transition_width > 0:
        kernel_x = np.arange(
            -transition_width,
            transition_width + 1,
        )

        sigma = max(
            1.0,
            transition_width / 2,
        )

        kernel = np.exp(
            -(
                kernel_x ** 2
            )
            / (
                2
                * sigma ** 2
            )
        )

        kernel /= kernel.sum()

        for k in range(K):
            A[:, k] = np.convolve(
                A[:, k],
                kernel,
                mode="same",
            )

    A /= (
        A.sum(
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    return A


def ar1_process(
    T,
    K,
    rho,
    rng,
):
    x = np.zeros(
        (T, K),
        dtype=float,
    )

    eps = rng.normal(
        0,
        1,
        size=(T, K),
    )

    x[0] = eps[0]

    innovation_scale = np.sqrt(
        max(
            1e-12,
            1.0 - rho ** 2,
        )
    )

    for t in range(
        1,
        T,
    ):
        x[t] = (
            rho * x[t - 1]
            + innovation_scale
            * eps[t]
        )

    return x


def add_correlated_secondary_recruitment(
    A,
    rng,
    secondary_weight=(
        SECONDARY_WEIGHT
    ),
    ar1_rho=AR1_RHO,
):
    T, K = A.shape

    background = ar1_process(
        T,
        K,
        ar1_rho,
        rng,
    )

    background = np.exp(
        background
    )

    background /= (
        background.sum(
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    mixed = (
        (
            1.0
            - secondary_weight
        )
        * A
        + secondary_weight
        * background
    )

    mixed /= (
        mixed.sum(
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    return mixed


def add_non_simplex_modulation(
    A,
    rng,
    global_gain_sd=(
        GLOBAL_GAIN_SD
    ),
    signed_deviation_sd=(
        SIGNED_DEVIATION_SD
    ),
    deviation_rho=(
        DEVIATION_RHO
    ),
):
    T, K = A.shape

    gain_latent = ar1_process(
        T,
        1,
        deviation_rho,
        rng,
    )[:, 0]

    gain = np.exp(
        global_gain_sd
        * gain_latent
        - 0.5
        * global_gain_sd ** 2
    )

    signed_dev = ar1_process(
        T,
        K,
        deviation_rho,
        rng,
    )

    A_eff = (
        gain[:, None] * A
        + signed_deviation_sd
        * signed_dev
    )

    # Deliberately no nonnegative projection
    # and no row-sum normalization here.
    return A_eff, gain


def simulate_metastable(
    seed,
    noise_sd=BASE_NOISE_SD,
    heterogeneity_scale=1.0,
    motifs_raw=None,
    mean_dwell=MEAN_DWELL,
    transition_width=(
        TRANSITION_WIDTH
    ),
    secondary_weight=(
        SECONDARY_WEIGHT
    ),
    ar1_rho=AR1_RHO,
):
    rng = np.random.default_rng(
        seed
    )

    if motifs_raw is None:
        motifs_raw = (
            TRUE_MOTIFS_RAW
        )

    motifs_raw = np.asarray(
        motifs_raw,
        dtype=float,
    )

    subjects = []
    state_sequences = []
    activations = []
    simplex_activations = []
    gains = []

    for s in range(
        N_SUBJECTS
    ):
        multiplicative = (
            1.0
            + rng.normal(
                0,
                BASE_MULTIPLICATIVE_SD
                * heterogeneity_scale,
                size=motifs_raw.shape,
            )
        )

        additive = rng.normal(
            0,
            BASE_ADDITIVE_SD
            * heterogeneity_scale,
            size=motifs_raw.shape,
        )

        subj_motifs = (
            motifs_raw
            * multiplicative
            + additive
        )

        states = (
            sample_state_sequence(
                T_PER_SUBJECT,
                N_TRUE_MOTIFS,
                rng,
                mean_dwell=(
                    mean_dwell
                ),
            )
        )

        A_state = (
            smooth_state_activations(
                states,
                N_TRUE_MOTIFS,
                transition_width=(
                    transition_width
                ),
            )
        )

        A_simplex = (
            add_correlated_secondary_recruitment(
                A_state,
                rng,
                secondary_weight=(
                    secondary_weight
                ),
                ar1_rho=(
                    ar1_rho
                ),
            )
        )

        (
            A_eff,
            gain,
        ) = (
            add_non_simplex_modulation(
                A_simplex,
                rng,
            )
        )

        X = (
            A_eff @ subj_motifs
        )

        X += rng.normal(
            0,
            noise_sd,
            size=X.shape,
        )

        subjects.append(X)
        state_sequences.append(
            states
        )
        activations.append(
            A_eff
        )
        simplex_activations.append(
            A_simplex
        )
        gains.append(gain)

    return {
        "subjects": subjects,
        "true_patterns": motifs_raw,
        "states": state_sequences,
        "activations": activations,
        "simplex_activations": (
            simplex_activations
        ),
        "gains": gains,
        "truth_kind": "point",
        "dynamics": {
            "mean_dwell": (
                mean_dwell
            ),
            "transition_width": (
                transition_width
            ),
            "secondary_weight": (
                secondary_weight
            ),
            "ar1_rho": (
                ar1_rho
            ),
        },
    }


In [ ]:
from pathlib import Path
from scipy.stats import bootstrap

RESULTS_CSV = Path(
    "simulation_pca_msaa_extended_outputs/"
    "pca_msaa_extended_results_paper.csv"
)

OUTPUT_DIR = Path("outputs/paper_figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_PNG = OUTPUT_DIR / "Figure3_PCA_MSAA_simulation_benchmark.png"
FIG_PDF = OUTPUT_DIR / "Figure3_PCA_MSAA_simulation_benchmark.pdf"

if not RESULTS_CSV.exists():
    raise FileNotFoundError(
        f"Missing {RESULTS_CSV}.\n"
        "Run revision_16_pca_msaa_clean_simulation_benchmark.ipynb "
        "with RUN_MODE='paper' first."
    )

results_df = pd.read_csv(RESULTS_CSV)
print("Loaded:", RESULTS_CSV)
print("Rows:", len(results_df))

In [ ]:
# Objectively preselected representative replicate from the final simulation
# package: paired MS-AA - PCA spatial-recovery difference closest to the median.
REPRESENTATIVE_SEED = 1941156788

sim = simulate_metastable(REPRESENTATIVE_SEED)

subjects, mean_subject_mean, sd = normalize_across_subjects(
    sim["subjects"]
)
true_patterns = transform_true_patterns(
    sim["true_patterns"],
    mean_subject_mean,
    sd,
    truth_kind=sim["truth_kind"],
)

X_stack = np.vstack(subjects)

msaa = fit_msaa(
    subjects,
    K=N_TRUE_MOTIFS,
    analysis_type="spatial",
    seed=REPRESENTATIVE_SEED,
    verbose=False,
)

S_group = np.mean(
    np.stack(
        [np.asarray(sub["S"], dtype=float) for sub in msaa["results_subj"]],
        axis=0,
    ),
    axis=0,
)

pca_fit = PCA(
    n_components=N_TRUE_MOTIFS,
    random_state=REPRESENTATIVE_SEED,
).fit(X_stack)

pca_patterns = np.asarray(pca_fit.components_, dtype=float)

aa_match = optimal_match(true_patterns, S_group, absolute=False)
pca_match = optimal_match(true_patterns, pca_patterns, absolute=True)

TRUE_ORDERED = true_patterns[aa_match["true_idx"]].copy()
AA_ORDERED = S_group[aa_match["recovered_idx"]].copy()

# Reorder PCA to the true-pattern order used above.
pca_by_true = {}
for true_idx, rec_idx in zip(
    pca_match["true_idx"],
    pca_match["recovered_idx"],
):
    row = pca_patterns[rec_idx].copy()
    r = np.corrcoef(true_patterns[true_idx], row)[0, 1]
    if np.isfinite(r) and r < 0:
        row *= -1
    pca_by_true[int(true_idx)] = row

PCA_ORDERED = np.stack(
    [pca_by_true[int(i)] for i in aa_match["true_idx"]],
    axis=0,
)

print("Representative seed:", REPRESENTATIVE_SEED)

In [ ]:

def mean_ci(x, n_boot=5000, seed=42):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]

    if len(x) == 0:
        return np.nan, np.array([np.nan, np.nan])

    rng = np.random.default_rng(seed)
    means = np.empty(n_boot, dtype=float)

    for b in range(n_boot):
        means[b] = rng.choice(
            x,
            size=len(x),
            replace=True,
        ).mean()

    return (
        float(x.mean()),
        np.quantile(
            means,
            [0.025, 0.975],
        ),
    )


# IMPORTANT:
# These are the actual regime names written by revision_16.
regime_order = [
    "PCA_control",
    "AA_control",
    "metastable",
]

regime_labels = {
    "PCA_control": "PCA-favorable",
    "AA_control": "AA-favorable",
    "metastable": "Metastable",
}


core = (
    results_df[
        results_df["regime"].isin(
            regime_order
        )
    ]
    .copy()
)


# ------------------------------------------------------------
# Guardrail: fail loudly if an expected control is missing
# ------------------------------------------------------------
counts = (
    core["regime"]
    .value_counts()
    .reindex(regime_order)
)

print("Core simulation counts:")
display(
    counts.to_frame("n")
)

missing_regimes = [
    regime
    for regime in regime_order
    if counts.loc[regime] == 0
]

if missing_regimes:
    raise ValueError(
        "The paper-mode checkpoint is missing expected regimes: "
        f"{missing_regimes}. "
        "Run revision_16_pca_msaa_clean_simulation_benchmark.ipynb "
        "in paper mode before making the figure."
    )


# ------------------------------------------------------------
# Panel A summaries
# ------------------------------------------------------------
rows = []

for r_i, regime in enumerate(
    regime_order
):
    sub = core[
        core["regime"] == regime
    ]

    for m_i, (method, col) in enumerate([
        (
            "MS-AA",
            "aa_motif_recovery",
        ),
        (
            "PCA",
            "pca_motif_recovery",
        ),
    ]):
        m, ci = mean_ci(
            sub[col],
            seed=42 + 100 * r_i + m_i,
        )

        rows.append({
            "regime": regime,
            "method": method,
            "mean": m,
            "lo": ci[0],
            "hi": ci[1],
        })

spatial_summary = pd.DataFrame(
    rows
)


# ------------------------------------------------------------
# Metastable-only data for Panels B/C
# ------------------------------------------------------------
meta = (
    core[
        core["regime"] == "metastable"
    ]
    .sort_values("rep")
    .copy()
)


# ------------------------------------------------------------
# Panel B: paired MS-AA - PCA spatial-recovery difference
# ------------------------------------------------------------
meta_delta = (
    meta["aa_motif_recovery"].to_numpy(dtype=float)
    - meta["pca_motif_recovery"].to_numpy(dtype=float)
)

delta_mean, delta_ci = mean_ci(
    meta_delta,
    seed=20260908,
)

print(
    "Metastable paired difference:",
    {
        "mean": delta_mean,
        "CI_low": float(delta_ci[0]),
        "CI_high": float(delta_ci[1]),
        "n": len(meta_delta),
        "fraction_MSAA_gt_PCA": float(
            np.mean(meta_delta > 0)
        ),
    },
)


In [ ]:

# ============================================================
# PAPER FIGURE 3 — corrected main simulation figure
# ============================================================

METHOD_COLORS = {
    "MS-AA": "#1f77b4",
    "PCA": "#ff7f0e",
}

fig = plt.figure(
    figsize=(13.2, 9.8)
)

gs = fig.add_gridspec(
    2,
    2,
    width_ratios=[1.0, 1.0],
    height_ratios=[1.0, 1.18],
    hspace=0.34,
    wspace=0.30,
)

axA = fig.add_subplot(
    gs[0, 0]
)

axB = fig.add_subplot(
    gs[0, 1]
)

axC = fig.add_subplot(
    gs[1, 0]
)

# D = three aligned heatmaps
subD = gs[1, 1].subgridspec(
    3,
    1,
    hspace=0.16,
)

axD1 = fig.add_subplot(
    subD[0, 0]
)

axD2 = fig.add_subplot(
    subD[1, 0]
)

axD3 = fig.add_subplot(
    subD[2, 0]
)


# ============================================================
# A — direct configuration recovery across three regimes
# ============================================================

x = np.arange(
    len(regime_order)
)

offset = 0.13


for method in [
    "MS-AA",
    "PCA",
]:

    cur = (
        spatial_summary[
            spatial_summary[
                "method"
            ] == method
        ]
        .set_index(
            "regime"
        )
    )

    xpos = (
        x - offset
        if method == "MS-AA"
        else x + offset
    )

    means = np.array(
        [
            cur.loc[
                regime,
                "mean",
            ]
            for regime
            in regime_order
        ],
        dtype=float,
    )

    lo = np.array(
        [
            cur.loc[
                regime,
                "lo",
            ]
            for regime
            in regime_order
        ],
        dtype=float,
    )

    hi = np.array(
        [
            cur.loc[
                regime,
                "hi",
            ]
            for regime
            in regime_order
        ],
        dtype=float,
    )

    yerr = np.vstack(
        [
            means - lo,
            hi - means,
        ]
    )

    # individual simulation replicates
    col = (
        "aa_motif_recovery"
        if method == "MS-AA"
        else "pca_motif_recovery"
    )

    for regime_i, regime in enumerate(
        regime_order
    ):
        vals = (
            core.loc[
                core[
                    "regime"
                ] == regime,
                col,
            ]
            .to_numpy(
                dtype=float
            )
        )

        jitter = np.linspace(
            -0.035,
            0.035,
            len(vals),
        )

        axA.scatter(
            np.full(
                len(vals),
                xpos[regime_i],
            )
            + jitter,
            vals,
            s=17,
            alpha=0.32,
            color=METHOD_COLORS[
                method
            ],
            edgecolors="none",
        )

    # mean + bootstrap CI
    axA.errorbar(
        xpos,
        means,
        yerr=yerr,
        color=METHOD_COLORS[
            method
        ],
        marker="o",
        markersize=7,
        linewidth=0,
        elinewidth=2.0,
        capsize=4,
        label=method,
        zorder=5,
    )


axA.set_xticks(
    x
)

axA.set_xticklabels(
    [
        regime_labels[
            r
        ]
        for r
        in regime_order
    ],
    rotation=12,
    ha="right",
)

axA.set_ylim(
    0,
    1.05,
)

axA.set_ylabel(
    "Direct spatial configuration recovery (r)"
)

axA.set_title(
    "Direct recovery depends on latent geometry",
    fontweight="bold",
)

axA.legend(
    frameon=False,
    loc="lower left",
)


# ============================================================
# B — paired metastable difference
# ============================================================

axB.axhline(
    0,
    color="0.55",
    linestyle="--",
    linewidth=1.2,
)

# individual paired differences
jitter = np.linspace(
    -0.035,
    0.035,
    len(meta_delta),
)

axB.scatter(
    jitter,
    meta_delta,
    s=24,
    alpha=0.35,
    color="#4C78A8",
)

# mean + bootstrap CI
axB.errorbar(
    [0],
    [delta_mean],
    yerr=np.array([
        [
            delta_mean
            - delta_ci[0]
        ],
        [
            delta_ci[1]
            - delta_mean
        ],
    ]),
    fmt="o",
    color="#1f77b4",
    markersize=8,
    elinewidth=2.2,
    capsize=5,
    zorder=5,
)

axB.set_xlim(
    -0.22,
    0.22,
)

# Keep a symmetric y-range around zero if possible.
delta_bound = max(
    0.16,
    1.25
    * np.nanmax(
        np.abs(
            meta_delta
        )
    ),
)

axB.set_ylim(
    -delta_bound,
    delta_bound,
)

axB.set_xticks(
    [0]
)

axB.set_xticklabels(
    [
        "MS-AA − PCA"
    ]
)

axB.set_ylabel(
    "Paired difference in spatial recovery (Δr)"
)

axB.set_title(
    "Metastable paired recovery difference",
    fontweight="bold",
)

axB.text(
    0.98,
    0.96,
    (
        f"mean Δr = {delta_mean:.3f}\n"
        f"95% CI [{delta_ci[0]:.3f}, {delta_ci[1]:.3f}]\n"
        f"{np.sum(meta_delta > 0)}/{len(meta_delta)} replicates > 0"
    ),
    transform=axB.transAxes,
    ha="right",
    va="top",
    fontsize=9,
)


# ============================================================
# C — individual configuration vs full latent span
# ============================================================

metrics = [
    (
        "Individual\nconfiguration",
        "aa_motif_recovery",
        "pca_motif_recovery",
    ),
    (
        "Latent span",
        "aa_span_recovery",
        "pca_span_recovery",
    ),
]

xx = np.arange(
    len(metrics)
)

for method, prefix, shift in [
    (
        "MS-AA",
        "aa",
        -0.12,
    ),
    (
        "PCA",
        "pca",
        0.12,
    ),
]:

    vals = []
    errs = []

    for metric_i, (
        label,
        aa_col,
        pca_col,
    ) in enumerate(
        metrics
    ):
        col = (
            aa_col
            if prefix == "aa"
            else pca_col
        )

        m, ci = mean_ci(
            meta[col],
            seed=1000
            + metric_i
            + (
                100
                if method == "PCA"
                else 0
            ),
        )

        vals.append(
            m
        )

        errs.append([
            m - ci[0],
            ci[1] - m,
        ])

    errs = np.asarray(
        errs
    ).T

    axC.errorbar(
        xx + shift,
        vals,
        yerr=errs,
        color=METHOD_COLORS[
            method
        ],
        marker="o",
        markersize=7,
        linewidth=1.8,
        capsize=4,
        label=method,
    )


axC.set_xticks(
    xx
)

axC.set_xticklabels(
    [
        m[0]
        for m
        in metrics
    ],
)

axC.set_ylabel(
    "Recovery (r)"
)

axC.set_ylim(
    0,
    1.05,
)

axC.set_title(
    "AA aligns individual motifs; PCA preserves the span",
    fontweight="bold",
)

axC.legend(
    frameon=False,
    loc="lower right",
)


# ============================================================
# D — representative metastable replicate
# ============================================================

def row_zscore(M):
    M = np.asarray(
        M,
        dtype=float,
    )

    return (
        M
        - M.mean(
            axis=1,
            keepdims=True,
        )
    ) / (
        M.std(
            axis=1,
            keepdims=True,
        )
        + 1e-8
    )


heatmap_panels = [
    (
        axD1,
        TRUE_ORDERED,
        "True configurations",
    ),
    (
        axD2,
        AA_ORDERED,
        "Matched MS-AA archetypes",
    ),
    (
        axD3,
        PCA_ORDERED,
        "Matched PCA components",
    ),
]

vmin = -2.5
vmax = 2.5

for ax, M, ylabel in heatmap_panels:

    ax.imshow(
        row_zscore(M),
        aspect="auto",
        interpolation="nearest",
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
    )

    ax.set_ylabel(
        ylabel,
        rotation=0,
        ha="right",
        va="center",
        labelpad=8,
    )

    ax.set_yticks([])

    ax.tick_params(
        axis="x",
        labelsize=8,
    )


axD1.set_xticklabels([])
axD2.set_xticklabels([])

axD3.set_xlabel(
    "Node / feature index"
)

axD1.set_title(
    (
        "Representative metastable replicate "
        f"(seed {REPRESENTATIVE_SEED})"
    ),
    fontweight="bold",
)


# ============================================================
# PANEL LABELS
# ============================================================

for ax, letter in zip(
    [
        axA,
        axB,
        axC,
        axD1,
    ],
    list(
        "ABCD"
    ),
):

    ax.text(
        -0.12,
        1.05,
        letter,
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top",
    )


# ============================================================
# FINAL FORMAT
# ============================================================

fig.suptitle(
    (
        "Simulation benchmark distinguishes motif recovery "
        "from span preservation"
    ),
    fontsize=16,
    fontweight="bold",
    y=0.995,
)

fig.subplots_adjust(
    top=0.93,
    bottom=0.08,
    left=0.09,
    right=0.98,
)

fig.savefig(
    FIG_PNG,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    FIG_PDF,
    bbox_inches="tight",
)

plt.show()

print(
    "Saved:",
    FIG_PNG,
)

print(
    "Saved:",
    FIG_PDF,
)


### Figure 3 interpretation

The simulation is a calibration, not a claim that brain activity literally
follows an archetypal generative model. PCA should dominate in the
PCA-favorable control, MS-AA should dominate in the AA-favorable control, and
the metastable regime tests the scientifically relevant distinction between
direct motif alignment and generic low-dimensional span preservation.